# EBI BioImage Archive — Upload Notebook

**Dataset:** 2026 Multicolour Bayer-SMLM paper  
**Source:** SMB share on intelliflash  
**Destination:** `ftp-private.ebi.ac.uk` (BioImage Archive Webin FTP)

## Workflow
1. **Configure paths** — set SMB root and output directory
2. **Scan + hash** — walk the share, MD5 every TIFF (~hours; run in tmux/screen)
3. **Inspect manifest** — preview `ebi_filelist.tsv` before uploading
4. **Upload** — FTP transfer with size-based resume
5. **Verify** — compare local manifest count to remote

> **Credentials:** enter Webin username/password in the *Upload* section when prompted.  
> Never hard-code or commit credentials.

---
## 0 — Imports

In [ ]:
import ftplib
import getpass
import hashlib
import os
import sys
import time
from datetime import timedelta
from pathlib import Path

try:
    from tqdm.notebook import tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False
    print("tqdm not available — progress will be printed every 50 files")

print("Python:", sys.version)

---
## 1 — Configuration

Edit the two paths below if needed; everything else derives from them.

In [ ]:
# ── Source (SMB share, mounted via GVFS on the analysis PC) ──────────────────
SMB_ROOT = Path(
    "/run/user/1000/gvfs/"
    "smb-share:server=intelliflash-mgmt-b.ch.private.cam.ac.uk,"
    "share=sycamore_asap_server/2026_Multicolour_Paper/Data"
)

# ── Output directory for manifest files ──────────────────────────────────────
# Writes next to this notebook by default
OUT_DIR = Path(".").resolve()

FILELIST_TSV  = OUT_DIR / "ebi_filelist.tsv"       # manifest for EBI portal
UPLOAD_PAIRS  = OUT_DIR / "ebi_upload_paths.txt"   # local→remote path pairs

# ── EBI path prefix (preserved inside the submission) ────────────────────────
EBI_PREFIX = "2026_Multicolour_Paper"

# ── FTP server ───────────────────────────────────────────────────────────────
FTP_HOST = "ftp-private.ebi.ac.uk"

# ── Sanity check ─────────────────────────────────────────────────────────────
if SMB_ROOT.exists():
    print(f"SMB share found: {SMB_ROOT}")
else:
    print(f"WARNING: SMB share not found at {SMB_ROOT}")
    print("Mount the share before running the scan cells.")

print(f"Output directory: {OUT_DIR}")

---
## 2 — Scan files and compute MD5 checksums

This will take **several hours** over SMB (I/O-bound).  
Run this cell in a `tmux` session or leave the notebook open.

Output files:
- `ebi_filelist.tsv` — `filename / md5 / size` manifest for EBI portal  
- `ebi_upload_paths.txt` — `local_path \t remote_path` pairs for the upload cell

In [ ]:
def md5sum(path: Path, chunk: int = 1 << 20) -> str:
    """Return hex MD5 of a file, reading in 1 MB chunks."""
    h = hashlib.md5()
    with path.open("rb") as f:
        while data := f.read(chunk):
            h.update(data)
    return h.hexdigest()


def human_size(n_bytes: int) -> str:
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if n_bytes < 1024:
            return f"{n_bytes:.1f} {unit}"
        n_bytes /= 1024
    return f"{n_bytes:.1f} PB"


def scan_and_hash(smb_root: Path, ebi_prefix: str,
                  filelist_out: Path, pathpairs_out: Path):
    """Walk smb_root, hash every TIFF, write manifest and path-pair files."""
    tifs = sorted(
        p for p in smb_root.rglob("*")
        if p.is_file() and p.suffix.lower() in {".tif", ".tiff"}
    )
    n = len(tifs)
    print(f"Found {n} TIFF files under {smb_root}")
    if n == 0:
        print("Nothing to do — is the SMB share mounted?")
        return

    total_bytes = sum(p.stat().st_size for p in tifs)
    print(f"Total size: {human_size(total_bytes)}")

    t0 = time.time()
    iterator = tqdm(enumerate(tifs, 1), total=n, unit="file") if HAS_TQDM \
               else enumerate(tifs, 1)

    with filelist_out.open("w") as fl, pathpairs_out.open("w") as pp:
        fl.write("filename\tmd5\tsize\n")
        for i, src in iterator:
            rel      = src.relative_to(smb_root)
            ebi_path = f"{ebi_prefix}/{rel}"
            size     = src.stat().st_size
            cksum    = md5sum(src)
            fl.write(f"{ebi_path}\t{cksum}\t{size}\n")
            pp.write(f"{src}\t{ebi_path}\n")

            if not HAS_TQDM and i % 50 == 0:
                elapsed  = time.time() - t0
                rate     = i / elapsed          # files/s
                eta_s    = (n - i) / rate if rate > 0 else 0
                print(f"  {i}/{n}  elapsed {timedelta(seconds=int(elapsed))}"
                      f"  ETA {timedelta(seconds=int(eta_s))}")

    elapsed = time.time() - t0
    print(f"\nDone in {timedelta(seconds=int(elapsed))}.")
    print(f"Manifest : {filelist_out}  ({filelist_out.stat().st_size:,} bytes)")
    print(f"Path pairs: {pathpairs_out}")


# ── Run ───────────────────────────────────────────────────────────────────────
scan_and_hash(SMB_ROOT, EBI_PREFIX, FILELIST_TSV, UPLOAD_PAIRS)

---
## 3 — Inspect the manifest

Run this after the scan to check file counts, directory breakdown, and total size before uploading.

In [ ]:
import csv
from collections import defaultdict

if not FILELIST_TSV.exists():
    print(f"Manifest not found: {FILELIST_TSV}")
    print("Run the scan cell first.")
else:
    rows = []
    with FILELIST_TSV.open() as f:
        reader = csv.DictReader(f, delimiter="\t")
        for row in reader:
            row["size"] = int(row["size"])
            rows.append(row)

    total_files = len(rows)
    total_size  = sum(r["size"] for r in rows)
    print(f"Total files : {total_files:,}")
    print(f"Total size  : {human_size(total_size)}")
    print()

    # Breakdown by top-level sub-directory (camera / dataset)
    by_dir = defaultdict(lambda: {"count": 0, "bytes": 0})
    for r in rows:
        # filename is  EBI_PREFIX/Camera/SubDir/...  — grab Camera/SubDir
        parts = Path(r["filename"]).parts
        key   = "/".join(parts[1:3]) if len(parts) >= 3 else parts[1]
        by_dir[key]["count"] += 1
        by_dir[key]["bytes"] += r["size"]

    print(f"{'Directory':<45}  {'Files':>6}  {'Size':>10}")
    print("-" * 65)
    for key in sorted(by_dir):
        d = by_dir[key]
        print(f"{key:<45}  {d['count']:>6}  {human_size(d['bytes']):>10}")

    print()
    print("First 5 rows:")
    print(f"  {'filename':<60}  {'md5':>32}  {'size':>12}")
    for r in rows[:5]:
        print(f"  {r['filename']:<60}  {r['md5']:>32}  {human_size(r['size']):>12}")

---
## 4 — FTP Upload

### Credentials
Enter your Webin username and password when prompted.  
Alternatively, set environment variables **before** launching Jupyter:
```bash
export WEBIN_USER="Webin-XXXXXX"
export WEBIN_PASS="your_password"
```

### Resume logic
Files already on the server at the correct size are **skipped**.  
Re-run the upload cell at any time to continue an interrupted transfer.

In [ ]:
# ── Credentials (env vars preferred; fall back to interactive prompt) ─────────
FTP_USER = os.environ.get("WEBIN_USER") or input("Webin username (e.g. Webin-12345): ").strip()
FTP_PASS = os.environ.get("WEBIN_PASS") or getpass.getpass("Webin password: ")
FTP_ROOT = f"/upload/{FTP_USER}"

print(f"Will upload to {FTP_HOST}{FTP_ROOT}/")

In [ ]:
UPLOAD_CHUNK = 1 << 20  # 1 MB per FTP block


def remote_size(ftp: ftplib.FTP, path: str) -> int:
    """Return remote file size, or -1 if the file does not exist."""
    try:
        return ftp.size(path)
    except ftplib.error_perm:
        return -1


def ensure_remote_dirs(ftp: ftplib.FTP, remote_path: str):
    """Recursively create all intermediate directories on the FTP server."""
    parts = Path(remote_path).parent.parts
    for i in range(1, len(parts) + 1):
        d = str(Path(*parts[:i]))
        try:
            ftp.mkd(d)
        except ftplib.error_perm:
            pass  # already exists — fine


def upload_file(ftp: ftplib.FTP, local: Path, remote: str) -> str:
    """Upload local → remote, skipping if sizes match. Returns 'skip'/'ok'/'error'."""
    local_size = local.stat().st_size
    if remote_size(ftp, remote) == local_size:
        return "skip"
    ensure_remote_dirs(ftp, remote)
    with local.open("rb") as f:
        ftp.storbinary(f"STOR {remote}", f, blocksize=UPLOAD_CHUNK)
    return "ok"


def load_upload_pairs(path: Path) -> list:
    pairs = []
    with path.open() as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            src, dst = line.split("\t", 1)
            pairs.append((Path(src), dst))
    return pairs


def run_upload(ftp_host: str, ftp_user: str, ftp_pass: str,
               ftp_root: str, pairs_file: Path):
    if not pairs_file.exists():
        print(f"Path-pairs file not found: {pairs_file}")
        print("Run the scan cell first.")
        return

    pairs = load_upload_pairs(pairs_file)
    n     = len(pairs)
    print(f"Uploading {n} files to {ftp_host} …")

    ftp = ftplib.FTP(ftp_host, timeout=60)
    ftp.login(ftp_user, ftp_pass)
    ftp.set_pasv(True)
    print("Login OK")

    counts = {"ok": 0, "skip": 0, "error": 0}
    t0     = time.time()

    iterator = tqdm(enumerate(pairs, 1), total=n, unit="file") if HAS_TQDM \
               else enumerate(pairs, 1)

    for i, (local, rel_remote) in iterator:
        remote = f"{ftp_root}/{rel_remote}"
        try:
            status = upload_file(ftp, local, remote)
        except Exception as exc:
            status = "error"
            print(f"  ERROR [{i}/{n}] {local.name}: {exc}", file=sys.stderr)
            # Attempt reconnect
            try:
                ftp.quit()
            except Exception:
                pass
            time.sleep(5)
            ftp = ftplib.FTP(ftp_host, timeout=60)
            ftp.login(ftp_user, ftp_pass)
            ftp.set_pasv(True)

        counts[status] += 1

        if not HAS_TQDM and i % 10 == 0:
            elapsed = time.time() - t0
            rate    = i / elapsed
            eta_s   = (n - i) / rate if rate > 0 else 0
            print(f"  [{i}/{n}]  ok={counts['ok']}  skip={counts['skip']}"
                  f"  err={counts['error']}"
                  f"  ETA {timedelta(seconds=int(eta_s))}")

    try:
        ftp.quit()
    except Exception:
        pass

    elapsed = time.time() - t0
    print(f"\nFinished in {timedelta(seconds=int(elapsed))}.")
    print(f"  Uploaded : {counts['ok']}")
    print(f"  Skipped  : {counts['skip']}")
    print(f"  Errors   : {counts['error']}")


# ── Run upload ────────────────────────────────────────────────────────────────
run_upload(FTP_HOST, FTP_USER, FTP_PASS, FTP_ROOT, UPLOAD_PAIRS)

---
## 5 — Verify remote file count

Compare the number of lines in the local manifest against the number of files visible on the FTP server.

> This uses a recursive `NLST` walk which can be slow for large trees.  
> The `lftp` one-liner in the comment below is faster if `lftp` is installed.

In [ ]:
# ── Count local manifest lines (excluding header) ─────────────────────────────
if FILELIST_TSV.exists():
    with FILELIST_TSV.open() as f:
        local_count = sum(1 for _ in f) - 1  # subtract header
    print(f"Local manifest: {local_count:,} files")
else:
    print("Manifest not found — run the scan cell first.")
    local_count = None


# ── Count remote files via FTP NLST ──────────────────────────────────────────
def count_remote_files(ftp: ftplib.FTP, remote_dir: str) -> int:
    """Recursively count files under remote_dir using NLST."""
    count = 0
    try:
        entries = ftp.nlst(remote_dir)
    except ftplib.error_perm:
        return 0
    for entry in entries:
        # Heuristic: entries with a '.' near the end are files
        if Path(entry).suffix:  # has an extension → likely a file
            count += 1
        else:  # directory — recurse
            count += count_remote_files(ftp, entry)
    return count


if local_count is not None:
    print("Connecting to FTP server for remote count …")
    ftp = ftplib.FTP(FTP_HOST, timeout=60)
    ftp.login(FTP_USER, FTP_PASS)
    ftp.set_pasv(True)

    remote_dir  = f"{FTP_ROOT}/{EBI_PREFIX}"
    remote_count = count_remote_files(ftp, remote_dir)
    ftp.quit()

    print(f"Remote files : {remote_count:,}")
    if remote_count == local_count:
        print("MATCH — all files present on server.")
    else:
        missing = local_count - remote_count
        print(f"MISMATCH — {missing:,} files missing from server. Re-run upload cell.")

---
## Alternative: lftp mirror (recommended if lftp is available)

Run the cell below in a terminal (not in the notebook) for a more robust upload.  
`lftp` handles resume, retries, and parallelism automatically.

```bash
lftp -u Webin-XXXXXX,YOUR_PASSWORD ftp-private.ebi.ac.uk <<'EOF'
set ftp:passive-mode true
mirror --reverse --verbose --parallel=4 \
  "/run/user/1000/gvfs/smb-share:server=intelliflash-mgmt-b.ch.private.cam.ac.uk,share=sycamore_asap_server/2026_Multicolour_Paper/Data" \
  /upload/Webin-XXXXXX/2026_Multicolour_Paper
bye
EOF
```

After the mirror completes, count remote files:

```bash
lftp -u Webin-XXXXXX,YOUR_PASSWORD ftp-private.ebi.ac.uk \
  -e "find /upload/Webin-XXXXXX/2026_Multicolour_Paper | wc -l; bye"
```

Compare against the local manifest:

```bash
# subtract 1 for the header line
echo "Local: $(( $(wc -l < ebi_filelist.tsv) - 1 )) files"
```

---
## Checklist

- [ ] EBI Webin account created and credentials to hand
- [ ] SMB share mounted at the path in cell 1
- [ ] Scan cell run → `ebi_filelist.tsv` + `ebi_upload_paths.txt` produced
- [ ] Manifest inspected (file count and size look right)
- [ ] MD5s spot-checked manually on a few files: `md5sum <file>`
- [ ] Upload complete (errors = 0)
- [ ] Remote count matches manifest count
- [ ] `ebi_filelist.tsv` submitted in the BioImage Archive portal
- [ ] Metadata fields completed in portal
- [ ] Accession number recorded in `claude/EBI_submission.md`: **S-BIADXXXXXXX**